In [ ]:
# Step 1: Install required packages (if not already installed)
#!pip install langchain langchain-google-genai langgraph requests python-dotenv

In [1]:
# Step 2: Import required libraries
import os
import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

In [2]:
# Load API key from .env file
load_dotenv()

True

In [3]:
# Step 3: Define REST API URLs
PRODUCT_API = "http://127.0.0.1:8000/products"
ORDER_API = "http://127.0.0.1:8002/orders"

In [4]:
# Step 4: Create Tool to fetch products from Product API
@tool
def get_products():
    """Get all product details from the Product API."""
    try:
        response = requests.get(PRODUCT_API, timeout=5)
        return response.json()
    except Exception as e:
        return f"Error connecting to Product API: {e}"

In [6]:
# Step 5: Create Tool to fetch orders from Order API
@tool
def get_orders():
    """Get all order details from the Order API."""
    try:
        response = requests.get(ORDER_API, timeout=5)
        return response.json()
    except Exception as e:
        return f"Error connecting to Order API: {e}"

In [7]:
# Step 6: Initialize Gemini Model and Tools list
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
        temperature = 0

)

tools = [get_products, get_orders]

In [8]:
# Step 7: Create Agent using LangGraph
agent = create_react_agent(
    model=llm,
    tools=tools
)

C:\Users\SMIT\AppData\Local\Temp\ipykernel_17864\3871573776.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [9]:
# Step 8: Define System Prompt (Strictly answer only products and orders)
system_message = """
You are a product and order chatbot.

You can ONLY answer questions about:
1. Products
2. Orders

For product questions, get the information from the Product API.
For order questions, get the information from the Order API.

Never make up product or order information.

If the user asks anything unrelated to products or orders, say:
"I can only help with product and order information."
"""

In [10]:
# Step 9: Test 1 - Ask about products
question = "What products are available?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': "I'm sorry, but I am currently unable to retrieve the product list because there was an error connecting to the Product API. Please try again later.", 'extras': {'signature': 'ErUDCrIDAWkUfROW10WS1QGlnl9Qqf0it/7b/Zknr7hXKgRE2WwOOcgHbaxH6Q7Ni4iCah7+eE1OcZuXgkCRCwR0395vRqrHXwkD+EaduxkQ2P/tpBMFZIJAhNXY7Li6HoLm4Qj/X4xkhyJSl0eqVWqTr56Wousy/h0+bE5tRnQnsTMtvuNw6WQ507HhIsf4oAAEWfJVpO+NxhbqFJVPs5rPV3tGYqBsbHDi89Fgth7xFSCmRl5KDS5P4f9iw2PpH1RQIt7eK+g/kg5LE2by3jHgMuvka8eC3NHsypXdcTDcUobUFNBppmZ98L7zBhPzOy20QbeEPrxNHVEN7Yyq+L6TDo8yUFYMetZym3NOhJAm5Pb7wg4ZMT+VGSpKGE/CiNEk6fBkxcHXwTJkk4bGvpTrlUv1YeMYSNmo7JcGG+bCe90OqQTCgkl2GHiGsuYds72BawoNvRiTFIUCQSFk0HCuqnOQPUrWa3hrn+itt0WoE2QbzFy02vBiikYnTQkcz+pAOuRmOlC2BBTIUunMCpaazniJ9nHopE01dEN9GzSmuYwsly7WgIArnmtvirvsUsi1riVJhjE='}}]


In [11]:
# Step 10: Test 2 - Ask about specific product details
question = "What is the price of iPhone 15 and is it in stock?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': "I'm sorry, but I am currently unable to retrieve product information due to a technical issue with the Product API. Please try again later.", 'extras': {'signature': 'EoMDCoADAWkUfRNFFYCciiS7+pBC2ICX+FdxSMda0B4RkARZqDINt1fp1bg+IymRmRVuO0KVT0M2HTMZr+QzazYTd6SFuybJ40v6xeXNUoXitLlEtopCVKe6tAV6G5WfaOvvqma/wHFd6gsJIjcr0O6rUzBa5wZvKnAmiKN1QD4HBTaWGbEHNg/fKYM5upxSqCtnw9kyEGJx9lo/fid7JMY9KbjbOvD/b2mbAYS3Yc76iVLpTRce8IrLupyUmsGxGUb+IUerEDjfnLjYvtucMk69bmnSfL0GpXyGap/VeQlGbyhu85hIZl9q8NTKV/N+jmWcaZXJNlvlh3wI4fHiTDtTx5dpdjjxi8iRhRP5tyY/aJSO8Ef1294ziIN2VPakSCjGqTaudKAItF0A47d5sJm6ubZK+nh9Z1VzaodHGpDAMVAl6U4xQ1S2dc27MtcI0/sUPoxTUIjlNuRhptmfXfz80/dCPsBLGCVhYIee6ZwR+OtjtL8CUscjcywACy/+L+IacoGT'}}]


In [12]:
# Step 11: Test 3 - Ask about order status
question = "What is the status of order 101?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': "I'm sorry, but I am currently unable to retrieve the status of order 101 because I cannot connect to the Order API. Please try again later.", 'extras': {'signature': 'EtQECtEEAWkUfRNtvmq385Qmmi5BrAI9yUFxtjvwTgJodsCEy+Q0+p36ubau3WOekmJ8o9jMp8BzIWjlCEsZi/7Owj0CazNw1xrkCWpBHOpqVJI7SmEEqAGfJdnwcYnoJa2duBzMILByxnlM2TXTAZIrlrSTsPFvfoxKTSypjOG8KuZiwPVpE46ts3tA4nmWWByMHMsbzzJKsy3EUUlCo1UibvHJSI3zR7KY1i8ujVREhtZ+hgJHVQ18kjhXsi9CbBDETQakELdshem2e6O01KWyPl1TfowE5xhmZDG8qlhNIxj1cyYuw3LXhENAsyN37G8xd/91IiEiRARoCOO1Fjel9fJlr5dKeYbqYMypg64NM4AM4NBEx6HwMSW3ZwwfO2zfPa7iSmmwG0c8TnFiwUdM4DERpJr9mKWVLh1dR7FXi1aSmC261dYy8eyX1u3vlmfiucJsPhl6Ugo5kwumDIvUUFDZr8ggd+++IbCCfbxIt3mFrnwgJ3xVhMGmYEi/v135pRdHHA+1RQU52MeRxYPXcYB/nIwS5Cw8HAxibaE3+XgEHCMzN17+deQVvTmHkiL3yKDMZGznLxUZ9YlhBUXlYMu+hb1EtCJ6CjzcP+/2VpFg7VogJLHeR6BUIL1JJ9Bw1ajRDMuP6Klss3beKrpdSvTw+O7LYdYsjpOCAHGrCcuJO3pK338yA80WtUexNC2X1/CT9GA0YW2KGbP8duLr28FqnjVgGYiZCWEnsXM9U+HNe2EN2TycP0wTa0meCfjwcOj+XYjaIWgy/MU8LK+LHseIciQ='}}]


In [13]:
# Step 12: Test 4 - Ask unrelated question (must be refused)
question = "What is the capital of India?"

result = agent.invoke({
    "messages": [
        ("system", system_message),
        ("user", question)
    ]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': 'I can only help with product and order information.', 'extras': {'signature': 'ErIFCq8FAWkUfRO/TU5JfzSRI0lfZ1IX+A9SwFhRfBlofT7vIlKUFlC38lu/ekiYYrHV7YxPec7eoPOH9HJ2klMbpJFneGyATd3yu6nPsg3AMqQytsVqQqyyK+YxQvSoKNxP6ak4Ooor+ZaDpL7Q1glzzpF0sK5x0oXOcSFZ/tqIk9t4cSXNKNmMDM3tfvcjujCf090lsa67Cqvw9HjJwTm2qWgKdUUd7wDH7UVEwUpKCnsvi8bjB2Q7TFYRF+gMQ91Kvhb1bVBKHsCfrMNcPUgumnnhmgdovboXHG/EVMFgdR7ft4vgZQ1H5X2f3OxkdX6xuAF03iSheoy6ZmNoXSZqSDXmyj+pqu/appujWs2lKKu171ZM+sN0GmI6EoNdVd9A+mOkV0bm/U12wQxFJ8NZuHvITL/yRGh3CsuufB5hw8rf4SNvoxhVD/4o2MfqK0Nu8OboU6E7tAceagF0LatjldMVGb4KCChZZN/v42t1suyhthIZd8IB69Ki4f2usdCrZrf7ptsBcQkWTYhEgIRJ8KvUwu/wnPjqSkTeYzPaq0R1Eh1NTzE83pF08cewTIAaBC+lImeaXY4R4M5wxAGu/7k+45rpaXRrS7pGSiQq/i78sli3IQbbogb1Ahd5q6lj7UMitV6EsrKgrrW4mL0xH/0xRBIixibo/w42o2qY4qT9+kcgSGuuGej9DFrS7h0Rh2+9B32TE+Nq9xYtLnN5jei34vM8rRyyPgA/3mnjMYh9YaxbTwtrOa45M33Cty+ujv6V4A7hrl4AaNU7VxRQCL1D3EgB+j6gJKZ2CcBPrd3itMaCAY8i+XXaU8uGdGhkEVKuEXaSM96g4ek3j1YeWLxcmtFRBuGSOtAxlmMAGnSONI69ePAvqQo0gg